In [5]:
!pip install -q fastparquet
!pip install -q statsforecast

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsforecast import StatsForecast
from statsforecast.models import TSB, AutoETS, AutoARIMA, Naive, SeasonalNaive, Theta, OptimizedTheta, CrostonOptimized, ADIDA, IMAPA, CrostonSBA, HoltWinters

%matplotlib inline

#####################################################
# Импортируем .py файлы с уже написанными функциями #
#####################################################

import sys
sys.path.append('/kaggle/input/datasets/faibus/diploma')

# функции для расчёта метрик
from metrics import (
    DEFAULT_METRICS,
    get_model_columns,
    compute_metrics_per_window,
    summarize_metrics,
    summarize_metrics_by_segments,
)

# функции для фильтрации и разделения рядов
from split_utils import filter_long_series, split_train_val_test

# функции для оценки и сбора предсказаний
from evaluation_utils import evaluate_frozen_windows

In [7]:
# данные о реальном спросе
real_demand = pd.read_parquet('/kaggle/input/datasets/faibus/diploma/real_demand_data.parquet', engine='fastparquet')

# выгружаем типы рядов
ts_dict_df = pd.read_csv('/kaggle/input/datasets/faibus/diploma/ts_dict_df')[['SKU_id', 'ts_type']]

In [8]:
# Выделяем SKU, принадлежащие к типу lumpy
lumpy_ts = list(ts_dict_df.query(" ts_type == 'smooth' ")['SKU_id'])

# фильтруем датасет по lumpy_ts
real_demand = real_demand.query(" SKU_id.isin(@lumpy_ts) ")

# причесываем датасет
real_demand = real_demand.rename(columns = {'date':'ds', 'real_demand':'y', 'SKU_id':'unique_id'})[['unique_id', 'ds', 'y']]

real_demand.shape

(730449, 3)

### Делим train / val / test

In [9]:
from split_utils import filter_long_series, split_train_val_test

real_demand_filtered = filter_long_series(
    real_demand,
    horizon=14,
    n_val_windows=1,
    n_test_windows=3,
    min_train_points=35,
    id_col="unique_id",
)

eval_df = real_demand_filtered[["unique_id", "ds", "y"]].sort_values(["unique_id", "ds"])

### Обучение

In [11]:
# 1. Создаем список с моделями
models = [
    SeasonalNaive(season_length=7, alias='SeasonalNaive7'),
    Naive(alias='Naive'),
    AutoETS(alias='AutoETS'),
    AutoARIMA(alias='AutoARIMA'),
    OptimizedTheta(alias='OptTheta'),
    HoltWinters(season_length=7, error_type="A", alias='HoltWinters_add_w_seas'),
    ADIDA(alias='ADIDA'),
    IMAPA(alias='IMAPA')
]

# обучаем только на train, val - подбор наилучших параметров, test - итоговое качество метрик
sf = StatsForecast(models=models, freq="D", n_jobs=1)
cv_frozen = sf.cross_validation(
    df=eval_df,
    h=14,
    step_size=14,
    n_windows=4,   # 1 val + 3 test
    refit=False,   # не переобучаем параметры
)

### Собираем прогнозы

In [12]:
val_cutoff = pd.Timestamp("2025-08-05")
test_cutoffs = [
    pd.Timestamp("2025-08-19"),
    pd.Timestamp("2025-09-02"),
    pd.Timestamp("2025-09-16"),
]

val_pred = cv_frozen[cv_frozen["cutoff"] == val_cutoff].copy()
test_pred = cv_frozen[cv_frozen["cutoff"].isin(test_cutoffs)].copy()

cutoff_to_window = {c: i + 1 for i, c in enumerate(test_cutoffs)}
test_pred["test_window"] = test_pred["cutoff"].map(cutoff_to_window)

### Считаем метрики

In [13]:
from metrics import DEFAULT_METRICS, get_model_columns, compute_metrics_per_window, summarize_metrics

val_model_cols = get_model_columns(
    val_pred,
    reserved_columns=("unique_id", "ds", "cutoff", "y", "index", "test_window"),
)
val_metrics_per_window = compute_metrics_per_window(val_pred, val_model_cols, DEFAULT_METRICS)
val_summary_mean, val_summary_stats = summarize_metrics(val_metrics_per_window)

test_model_cols = get_model_columns(
    test_pred,
    reserved_columns=("unique_id", "ds", "cutoff", "y", "index", "test_window"),
)
test_metrics_per_window = compute_metrics_per_window(test_pred, test_model_cols, DEFAULT_METRICS)
test_summary_mean, test_summary_stats = summarize_metrics(test_metrics_per_window)

print("VAL mean metrics:")
display(val_summary_mean)

print('')

print("TEST mean metrics (3 windows x 14 days):")
display(test_summary_mean)

VAL mean metrics:


,mae,rmse,smape,wape
model,,,,
ADIDA,4.759541,10.874430,52.758238,37.927995
AutoARIMA,4.780327,10.468426,52.702471,38.093636
AutoETS,4.864460,10.781937,53.535226,38.764075
HoltWinters_add_w_seas,4.926053,11.120031,54.560889,39.254899
IMAPA,4.759678,10.873626,52.758539,37.929086
Naive,5.828428,11.780555,66.497392,46.445779
OptTheta,4.882597,10.883224,53.447438,38.908604
SeasonalNaive7,5.921021,12.950117,68.645329,47.183634



TEST mean metrics (3 windows x 14 days):


,mae,rmse,smape,wape
model,,,,
ADIDA,4.711835,9.725875,53.349128,37.930767
AutoARIMA,4.645324,9.430573,53.210168,37.379712
AutoETS,4.897960,10.650955,54.049763,39.443231
HoltWinters_add_w_seas,4.843533,10.419189,54.716840,38.998031
IMAPA,4.713971,9.728193,53.381173,37.948226
Naive,5.597424,11.213379,65.044892,45.073219
OptTheta,4.862125,10.558707,54.025230,39.152104
SeasonalNaive7,5.804655,11.842730,66.689998,46.744395


In [14]:
test_pred.to_csv("smooth_forecast.csv")